# Report 2: learned Cropr token pruning

All model loading, normalization, task heads and Cropr construction live in
`cropr.py`. The runner constructs one model/task at a time. Existing datasets,
weights and Report 1 are read only; all results go into `report_2/cropr measurements`.

Classification uses one auxiliary query per stage and allows pruning without
fusion. Segmentation uses one query per patch-grid position and requires
last-layer fusion to restore full token count and original spatial order.
Both train with main task loss plus an auxiliary loss that teaches the scorer.
The pretrained backbone stays frozen. No cached features or LoRA are used.

The measurement methods and report panels match Report 1. LoRA/cache-specific
fields are N/A; Cropr/token fields are additional. One pruning variant per run.


In [1]:
import sys
import os
import importlib
from pathlib import Path

# Run from the project root or report_2 directory.
_here = Path.cwd().resolve()
REPORT2_DIR = _here if (_here / "cropr.py").is_file() else _here / "report_2"
if not (REPORT2_DIR / "cropr.py").is_file():
    raise FileNotFoundError("Open this notebook from the project root or report_2")
sys.path.insert(0, str(REPORT2_DIR)) if str(REPORT2_DIR) not in sys.path else None
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

import torch
import cropr
import report2_cropr_runner as runner
importlib.reload(cropr)
importlib.reload(runner)
ROOT = cropr.resolve_project_root(REPORT2_DIR.parent)
torch.set_num_threads(1)  # Same CPU-thread setting as Report 1.
print("Model setup:", cropr.__file__)
print("Runner:", runner.__file__)
print("Dataset/model root:", ROOT)


Model setup: D:\School Stuff\shanghai work\Deep Learning Project\report_2\cropr.py
Runner: D:\School Stuff\shanghai work\Deep Learning Project\report_2\report2_cropr_runner.py
Dataset/model root: D:\School Stuff\shanghai work\Deep Learning Project


c:\ProgramData\anaconda3\envs\cleanjupyter\lib\python\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Experiment settings
Choose models, tasks and one pruning variant.


In [ ]:
# Same datasets, resolution, epoch budgets, head LR and measurement settings as Report 1.
SEED = 42
SIZE = 448
MODEL_NAMES = ("radio",)  #"dinov2", "siglip2", "radio"
TASKS = ("classification", "segmentation")    #"classification", "segmentation"
VARIANT = "gradual_llf"  # "gradual_llf", "three_stage_llf", "gradual_no_llf"
# For gradual_no_llf, set TASKS = ("classification",). Segmentation requires fusion.
PRUNING_RATE = 8         # Patch tokens removed per gradual stage; not a percentage.
CROPR_LR = 1e-4
LORA_R = 8
LORA_ALPHA = 16.0
LORA_LR = 1e-4
LORA_TARGETS = None
AUX_WEIGHT = 1.0
NUM_HEADS = 1

EPOCHS = {"classification": 2, "segmentation": 2}
LR = {"classification": 1e-3, "segmentation": 1e-3}
WEIGHT_DECAY = 0.01
TRAIN_PRECISION = "bf16"
MAX_GRAD_NORM = 1.0
# Smaller physical batches than Report 1; effective training batch remains 16.
# Segmentation auxiliary queries require extra memory. No automatic OOM retry.
MICRO_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
EVAL_BATCH_SIZE = 1
BATCH_SIZE = 2           # Benchmark batches 1 and 2, as in Report 1.
NUM_WORKERS = 4
LOADER_BACKEND = "thread"
FAIL_FAST = True
EVAL_TF32 = False
BENCH_WARMUP = 10
BENCH_REPEATS = 50
BENCH_TRIALS = 3
FLOP_BATCHES = 3
PROFILE_REPEATS = 3
print("Experiments:", [(task, name, VARIANT) for task in TASKS for name in MODEL_NAMES])
print("Effective training batch:", MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS)


Experiments: [('classification', 'dinov2', 'gradual_llf'), ('segmentation', 'dinov2', 'gradual_llf')]
Effective training batch: 16


## Training run
This cell starts the full selected run.


In [17]:
# TRAINING RUN ? execute this cell to train, evaluate, save checkpoints and measure.
# This uses the existing ImageNet/ADE20K saved splits; it does not recreate them.
results, OUT = runner.run_cropr(
    ROOT=ROOT, SEED=SEED, SIZE=SIZE, MODEL_NAMES=MODEL_NAMES, TASKS=TASKS,
    VARIANT=VARIANT, PRUNING_RATE=PRUNING_RATE, NUM_HEADS=NUM_HEADS,
    CROPR_LR=CROPR_LR, AUX_WEIGHT=AUX_WEIGHT,
    LORA_R=LORA_R, LORA_ALPHA=LORA_ALPHA, LORA_LR=LORA_LR, LORA_TARGETS=LORA_TARGETS,
    EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY,
    TRAIN_PRECISION=TRAIN_PRECISION, MAX_GRAD_NORM=MAX_GRAD_NORM,
    MICRO_BATCH_SIZE=MICRO_BATCH_SIZE, GRAD_ACCUM_STEPS=GRAD_ACCUM_STEPS,
    EVAL_BATCH_SIZE=EVAL_BATCH_SIZE, BATCH_SIZE=BATCH_SIZE,
    NUM_WORKERS=NUM_WORKERS, LOADER_BACKEND=LOADER_BACKEND, FAIL_FAST=FAIL_FAST,
    EVAL_TF32=EVAL_TF32, BENCH_WARMUP=BENCH_WARMUP, BENCH_REPEATS=BENCH_REPEATS,
    BENCH_TRIALS=BENCH_TRIALS, FLOP_BATCHES=FLOP_BATCHES, PROFILE_REPEATS=PROFILE_REPEATS,
)
print("Run folder:", OUT)
print("Checkpoints: best_cropr.pt / last_cropr.pt (LoRA adapters + task head + all learned pruners)")
print("Measurements:", OUT / "cropr measurements.csv")


Validate classification/val: 100%|██████████| 50000/50000 [00:06<00:00, 7519.11files/s]


classification {'train': 100000, 'dev': 20000, 'val': 50000}


Validate segmentation/val: 100%|██████████| 2000/2000 [00:00<00:00, 7801.42files/s]


segmentation {'train': 18189, 'dev': 2021, 'val': 2000}
Results: D:\School Stuff\shanghai work\Deep Learning Project\report_2\cropr measurements\20261020_105938_914022
Cropr variant: gradual_llf | precision: bf16

Starting classification: siglip2


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Evaluate classification: 100%|██████████| 20000/20000 [10:28<00:00, 31.80batch/s, load mean=0.002s]


...

  main=0.8871, aux=20.9621 | dev miou_pct=38.57 | peak=2.00 GiB


Evaluate segmentation: 100%|██████████| 2021/2021 [01:09<00:00, 28.95batch/s, load mean=0.002s]


  main=0.6166, aux=17.1763 | dev miou_pct=42.96 | peak=2.00 GiB


Evaluate segmentation: 100%|██████████| 2000/2000 [01:09<00:00, 28.64batch/s, load mean=0.002s]



Finished.  Measurements saved to:
D:\School Stuff\shanghai work\Deep Learning Project\report_2\cropr measurements\20261020_105938_914022\cropr measurements.csv
classification siglip2 → completed
classification radio → completed
segmentation siglip2 → completed
segmentation radio → completed
Run folder: D:\School Stuff\shanghai work\Deep Learning Project\report_2\cropr measurements\20261020_105938_914022
Checkpoints: best_cropr.pt / last_cropr.pt (task head + all learned pruners)
Measurements: D:\School Stuff\shanghai work\Deep Learning Project\report_2\cropr measurements\20261020_105938_914022\cropr measurements.csv


## Report metrics ? same panels as Report 1


In [ ]:
from pathlib import Path
# REPORT METRICS ? no training in the cells below.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
TRAINING_MODE = "cropr"

# Set RUN_FOLDER to an explicit folder to inspect another experiment.
# Otherwise use this session's OUT, then the latest run of the selected mode.
# RUN_FOLDER = Path(ROOT) / "report_2" / "cropr measurements" / "combined_20261020"
RUN_FOLDER = Path(ROOT) / "report_2" / "cropr measurements" / "combined_20261020"


def load_selected_run(folder=None):
    if folder is None:
        folder = globals().get("OUT")
    if folder is None:
        files = sorted((ROOT / "report_2" / "cropr measurements").glob("*/cropr measurements.csv"))
        expected_mode = globals().get("TRAINING_MODE")
        for csv_path in reversed(files):
            config_path = csv_path.parent / "configuration.json"
            if not config_path.is_file():
                continue
            candidate = json.loads(config_path.read_text(encoding="utf-8"))
            if expected_mode is None or candidate.get("training_mode") == expected_mode:
                folder = csv_path.parent
                break
    if folder is None:
        raise FileNotFoundError("No results for the selected training mode.  Set RUN_FOLDER explicitly.")
    run_path = Path(folder).expanduser().resolve()
    config = json.loads((run_path / "configuration.json").read_text(encoding="utf-8"))
    raw = pd.read_csv(run_path / "cropr measurements.csv")
    completed = raw.loc[raw["status"].eq("completed")].copy()
    if completed.empty:
        raise ValueError(f"No completed experiments in {run_path}.  Inspect the runner's error files.")
    if "training_mode" not in completed:
        completed["training_mode"] = config.get("training_mode", "unknown")
    modes = completed["training_mode"].dropna().unique()
    if len(modes) != 1 or modes[0] != "cropr":
        raise ValueError("Select one Cropr run at a time.")
    mode = str(modes[0])
    if config.get("training_mode", mode) != mode:
        raise ValueError("Results and run configuration disagree about training mode.")
    if mode == "lora":
        if "inference_lora_merged" not in completed or not completed["inference_lora_merged"].astype(str).str.lower().isin(["true", "1", "1.0"]).all():
            raise ValueError("LoRA inference rows must record merged adapters.")
    if completed.duplicated(["task", "model"]).any():
        raise ValueError("Duplicate task/model rows found.")
    return run_path, config, raw, completed, mode


run, run_config, results, df, training_mode = load_selected_run(RUN_FOLDER)
mode_label = "Cropr: " + run_config["cropr_variant"]
print("Results:", run)
print("Training mode:", training_mode)
print(f"Completed experiments: {len(df)} / {len(results)}")

models = ["dinov2", "siglip2", "radio"]
model_labels = ["DINOv2", "SigLIP 2", "RADIO"]
tasks = ["classification", "segmentation"]
task_labels = ["ImageNet", "ADE20K"]
colors = ["#377eb8", "#ff8c42"]

batches = pd.to_numeric(df["batch_size"], errors="coerce").dropna().unique()
if len(batches) != 1:
    raise ValueError("Use the same batch size across experiments.")
batch = int(batches[0])

resolutions = pd.to_numeric(
    df["resolution"], errors="coerce"
).dropna().unique()
if len(resolutions) != 1:
    raise ValueError("Use the same resolution across experiments.")
size = int(resolutions[0])

df["training_minutes"] = pd.to_numeric(
    df["training_seconds"], errors="coerce"
) / 60

WHOLE = "whole_model_forward_gflops_per_image"
ATTENTION = "attention_forward_gflops_per_image"
GMACS = "whole_model_forward_gmacs_per_image"

coverage_flags = {
    WHOLE: "whole_model_flops_coverage_ok",
    ATTENTION: "attention_flops_coverage_ok",
    GMACS: "whole_model_flops_coverage_ok",
}


def true_flags(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "1.0"])
    )


def values(task, metric):
    subset = df.loc[df["task"] == task].set_index("model")

    if metric not in subset:
        return np.full(len(models), np.nan)

    data = pd.to_numeric(subset[metric], errors="coerce")

    flag = coverage_flags.get(metric)
    if metric.endswith("_gpu_time_pct"):
        flag = "component_gpu_attribution_available"

    if flag is not None:
        if flag not in subset:
            return np.full(len(models), np.nan)
        data = data.where(true_flags(subset[flag]))

    return data.reindex(models).to_numpy(dtype=float)


def setup_axis(ax, title, ylabel):
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xticks(np.arange(len(models)), model_labels)
    ax.set_xlim(-0.6, len(models) - 0.4)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def bars(ax, x, y, color, label, errors=None):
    valid = np.isfinite(y)
    if not valid.any():
        return []

    plotted = ax.bar(
        x[valid], y[valid], width=0.35,
        color=color, label=label,
    )
    ax.bar_label(
        plotted,
        labels=[f"{v:.2f}" for v in y[valid]],
        padding=3, fontsize=8,
    )

    upper = y[valid].tolist()

    if errors is not None:
        valid_error = valid & np.isfinite(errors) & (errors >= 0)
        if valid_error.any():
            ax.errorbar(
                x[valid_error], y[valid_error],
                yerr=errors[valid_error],
                fmt="none", ecolor="black", capsize=4,
            )
            upper.extend((y[valid_error] + errors[valid_error]).tolist())

    return upper


def comparison(ax, metric, title, ylabel, error_metric=None):
    setup_axis(ax, title, ylabel)
    upper = []

    for i, task in enumerate(tasks):
        x = np.arange(len(models)) + (i - 0.5) * 0.36
        errors = values(task, error_metric) if error_metric else None
        upper.extend(
            bars(
                ax, x, values(task, metric),
                colors[i], task_labels[i], errors,
            )
        )

    maximum = max(upper) if upper else 0
    ax.set_ylim(0, maximum * 1.25 if maximum > 0 else 1)

    if upper:
        ax.legend(fontsize=8)
    else:
        message = (
            "Missing / coverage not verified"
            if metric in coverage_flags else "Not available"
        )
        ax.text(
            0.5, 0.5, message,
            transform=ax.transAxes,
            ha="center", va="center", fontsize=9,
        )


fig, axes = plt.subplots(5, 3, figsize=(18, 23))
axes = axes.ravel()

# Accuracy.
accuracy_panels = [
    (
        axes[0], "classification",
        "ImageNet classification accuracy",
        [
            ("heldout_top1_pct", "Top-1", "#377eb8"),
            ("heldout_top5_pct", "Top-5", "#80b1d3"),
        ],
    ),
    (
        axes[1], "segmentation",
        f"ADE20K segmentation accuracy at {size}×{size}",
        [
            ("heldout_miou_pct", "mIoU", "#ff8c42"),
            ("heldout_pixel_accuracy_pct", "Pixel accuracy", "#fdbf6f"),
        ],
    ),
]

for ax, task, title, metrics in accuracy_panels:
    setup_axis(ax, title, "Percent ↑")
    for i, (metric, label, color) in enumerate(metrics):
        x = np.arange(len(models)) + (i - 0.5) * 0.36
        bars(ax, x, values(task, metric), color, label)
    ax.set_ylim(0, 110)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)

# Speed, memory, and corrected forward counts.
panels = [
    (
        "b1_latency_median_ms",
        "Median GPU-stream latency — batch 1",
        "Milliseconds ↓", None,
    ),
    (
        "b1_latency_p95_ms",
        "95th-percentile GPU-stream latency — batch 1",
        "Milliseconds ↓", None,
    ),
    (
        f"b{batch}_images_per_second",
        f"CUDA-event throughput — batch {batch}",
        "Images / second ↑",
        f"b{batch}_images_per_second_std",
    ),
    (
        f"b{batch}_wall_images_per_second",
        f"Wall-clock model throughput — batch {batch}",
        "Images / second ↑",
        f"b{batch}_wall_images_per_second_std",
    ),
    (
        "training_minutes",
        "Training time — excludes validation",
        "Minutes ↓", None,
    ),
    (
        "train_peak_allocated_gib",
        "Peak allocated GPU memory — training",
        "GiB ↓", None,
    ),
    (
        f"b{batch}_peak_allocated_gib",
        f"Peak allocated GPU memory — inference batch {batch}",
        "GiB ↓", None,
    ),
    (
        WHOLE,
        "Whole-model forward — matrix/conv convention",
        "GFLOPs / image ↓", None,
    ),
    (
        ATTENTION,
        "Attention forward — projections + matrix products",
        "GFLOPs / image ↓", None,
    ),
    (
        GMACS,
        "Whole-model forward — GMACs",
        "GMACs / image ↓", None,
    ),
]

for ax, (metric, title, ylabel, error_metric) in zip(axes[2:12], panels):
    comparison(ax, metric, title, ylabel, error_metric)

# GPU-time breakdown.
parts = ["attention", "mlp", "head", "other"]
part_colors = ["#756bb1", "#31a354", "#e6550d", "#bdbdbd"]

for ax, task, label in zip(axes[12:14], tasks, task_labels):
    setup_axis(ax, f"{label}: forward GPU time breakdown", "Time share (%)")

    data = np.array([
        values(task, f"{part}_gpu_time_pct") for part in parts
    ])
    valid = np.isfinite(data).all(axis=0)
    x = np.arange(len(models))[valid]
    bottom = np.zeros(valid.sum())

    if valid.any():
        for i, part in enumerate(parts):
            heights = data[i, valid]
            ax.bar(
                x, heights, bottom=bottom, width=0.55,
                color=part_colors[i],
                label="MLP" if part == "mlp" else part.title(),
            )
            for position, height, base in zip(x, heights, bottom):
                if height >= 7:
                    ax.text(
                        position, base + height / 2, f"{height:.0f}%",
                        ha="center", va="center", fontsize=9,
                    )
            bottom += heights

        ax.legend(fontsize=8, ncol=2)

        for position in np.arange(len(models))[~valid]:
            ax.text(position, 3, "N/A", ha="center", color="gray")
    else:
        ax.text(
            0.5, 0.5, "GPU attribution unavailable",
            transform=ax.transAxes, ha="center", va="center",
        )

    ax.set_ylim(0, 115)

axes[14].set_visible(False)

fig.suptitle(
    f"{mode_label} baseline measurements\n"
    "↑ Higher is better   |   ↓ Lower is better",
    fontsize=17,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

print("Training time scope:", run_config.get("training_seconds_scope", "See result.json"))
print("Inference uses the RGB Cropr model with auxiliary aggregation/heads skipped.")
print(
    "Forward counts: matrix multiplications, convolutions, and dense attention products.\n"
    "Convention: 2 FLOPs per MAC; GMACs = GFLOPs / 2.\n"
    "Excludes softmax, normalization, activations, interpolation, pooling, "
    "sorting, and elementwise work.\n"
    "Coverage flags refer to this convention, not every arithmetic instruction.\n"
    "Legacy partial profiler FLOPs are not used in these compute plots.\n"
    "Throughput error bars show sample standard deviation across timing trials.\n"
    "With one timing trial, no standard-deviation error bar is available.\n"
    "Inference timing excludes data loading and GPU transfers.\n"
    "Wall-clock throughput includes CPU launch overhead.\n"
    "GPU-time shares are not FLOP shares.\n"
    "Compare training time only with matching epoch budgets and dataset sizes."
)

for metric in (WHOLE, ATTENTION):
    if not any(np.isfinite(values(task, metric)).any() for task in tasks):
        print(
            f"\nMissing or unverified: {metric}\n"
            "Inspect the saved FLOP coverage and error files.\n"
            "For an updated run, inspect forward_flops.json, flop_samples/, "
            "and any FLOP error file."
        )

In [19]:
import pandas as pd
from IPython.display import display

# Uses the run selected in cell 4.
run, run_config, all_saved_fields, df, training_mode = load_selected_run(RUN_FOLDER)

model_names = {
    "dinov2": "DINOv2",
    "siglip2": "SigLIP 2",
    "radio": "RADIO",
}

model_tables = {}

for model, rows in df.groupby("model", sort=False):
    batches = rows["batch_size"].dropna().unique()
    if len(batches) != 1:
        raise ValueError("Each model table requires one benchmark batch size.")
    batch = int(batches[0])

    columns = {
        "task": "Task",

        # Held-out evaluation
        "heldout_loss": "Test loss",
        "heldout_top1_pct": "Top-1 (%)",
        "heldout_top5_pct": "Top-5 (%)",
        "heldout_miou_pct": "mIoU (%)",
        "heldout_pixel_accuracy_pct": "Pixel accuracy (%)",
        

        # Validation-based checkpoint selection
        
        # Compute and speed
        "whole_model_forward_gflops_per_image": "GFLOPs/image",
        f"b{batch}_wall_images_per_second": f"Images/s (batch {batch})",
        "b1_latency_median_ms": "Batch-1 latency (ms)",

        # Pruning
        "cropr_input_patch_tokens": "Original patches",
        "cropr_removed_patch_tokens": "Pruned patches",
        "cropr_min_patch_tokens": "Minimum patches",
        "cropr_final_patch_tokens": "Final patches",
        "cropr_token_processing_reduction_pct": "Token processing saved (%)",
    }

    table = rows.reindex(columns=columns).rename(columns=columns).copy()

    # Do not display FLOPs whose measurement coverage was not verified.
    coverage = rows.get(
        "whole_model_flops_coverage_ok",
        pd.Series(False, index=rows.index),
    )
    verified = coverage.astype(str).str.lower().isin(["true", "1", "1.0"])
    table.loc[~verified, "GFLOPs/image"] = float("nan")

    table = table.reset_index(drop=True)
    model_tables[model] = table

    display(
        table.style
        .hide(axis="index")
        .format(precision=2, na_rep="N/A")
        .set_caption(model_names.get(model, model))
    )

print("Top-1: classification accuracy. mIoU: segmentation quality.")
print("Throughput excludes data loading and GPU transfer.")
print("Fusion restores pruned patches before the final block.")
print("Token processing saved is not a measure of information lost.")

Task,Test loss,Top-1 (%),Top-5 (%),mIoU (%),Pixel accuracy (%),GFLOPs/image,Images/s (batch 2),Batch-1 latency (ms),Original patches,Pruned patches,Minimum patches,Final patches,Token processing saved (%)
classification,2.14,51.12,76.71,N/A,N/A,150.37,36.40,27.69,784,80,704,784,4.68
segmentation,0.91,N/A,N/A,27.40,73.39,151.66,34.19,29.07,784,80,704,784,4.68


Task,Test loss,Top-1 (%),Top-5 (%),mIoU (%),Pixel accuracy (%),GFLOPs/image,Images/s (batch 2),Batch-1 latency (ms),Original patches,Pruned patches,Minimum patches,Final patches,Token processing saved (%)
classification,1.22,71.11,92.24,N/A,N/A,150.30,33.58,30.42,784,80,704,784,4.63
segmentation,0.65,N/A,N/A,42.03,80.01,151.60,37.66,27.27,784,80,704,784,4.63


Top-1: classification accuracy. mIoU: segmentation quality.
Throughput excludes data loading and GPU transfer.
Fusion restores pruned patches before the final block.
Token processing saved is not a measure of information lost.


### Interpreting and reloading the run

The charts and tables above use Report 1's metric names, scales, coverage flags,
and plot definitions. Missing/unsupported measurements remain N/A, never zero.
LoRA-specific entries are N/A because this experiment trains Cropr instead.
`all_saved_fields` retains every saved metric. Per-epoch main/auxiliary losses
are in each experiment's `epochs.csv`; per-block token counts are in `tokens.json`.

Timing uses FP32, TF32 disabled, batch 1 and the configured benchmark batch;
training uses BF16 by default. The effective training batch is 16, but the
physical training and evaluation batches are smaller than Report 1's current
notebook. Account for this when comparing training throughput/time. Pruning and
auxiliary supervision also change the objective, so this is not an exact repeat
of the LoRA training experiment.

To inspect an earlier run without retraining, run the imports cell, then set
`RUN_FOLDER` in the plots cell to its saved folder and run the report cells.
To reload an experiment for inference:

```python
model = cropr.load_cropr_checkpoint(
    RUN_FOLDER / "segmentation_dinov2" / "best_cropr.pt",
    root=ROOT, device="cuda",
)
```

Checkpoints restore the task head and pruners against verified local pretrained
weights. Optimizer-state resume is not supported. Training starts fresh on each
run. The local PyTorch environment previously showed a native shutdown issue;
full dataset training has not been executed by this edit.
